# Practical 5: Ensemble Learning - Random Forest for Car Safety / Acceptability
**Problem Statement:** Implement a Random Forest Classifier to predict the acceptability of cars based on buying price, maintenance cost, number of doors, passenger capacity, luggage boot size and safety level.

**Dataset:** https://www.kaggle.com/datasets/elikplim/car-evaluation-data-set (file: `car_evaluation.csv`)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

## 2. Load the Dataset
The Kaggle file has no header row, so column names are assigned manually. Place `car_evaluation.csv` in the same folder as this notebook.

In [ ]:
col_names = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]

df = pd.read_csv("car_evaluation.csv", header=None)

# If the file already has a header row, remove it
if str(df.iloc[0, -1]).strip().lower() == "class":
    df = df.iloc[1:].reset_index(drop=True)

df.columns = col_names
df.head()

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

## 3. Explore the Data
All columns are categorical. Here are the unique values and how the target classes are distributed.

In [ ]:
for col in df.columns:
    print(f"{col:9s}: {df[col].unique().tolist()}")

In [ ]:
print(df["class"].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(x="class", data=df, order=df["class"].value_counts().index, palette="viridis")
plt.title("Car Acceptability Classes")
plt.show()

In [ ]:
# How safety and persons relate to the class
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(x="safety", hue="class", data=df, ax=axes[0], palette="Set2")
axes[0].set_title("Safety vs Class")
sns.countplot(x="persons", hue="class", data=df, ax=axes[1], palette="Set2")
axes[1].set_title("Persons vs Class")
plt.tight_layout()
plt.show()

## 4. Encode the Categorical Features
The categories have a natural order (low < med < high etc.), so ordinal encoding with an explicit order is used. The target is label encoded.

In [ ]:
order = {
    "buying":   ["low", "med", "high", "vhigh"],
    "maint":    ["low", "med", "high", "vhigh"],
    "doors":    ["2", "3", "4", "5more"],
    "persons":  ["2", "4", "more"],
    "lug_boot": ["small", "med", "big"],
    "safety":   ["low", "med", "high"],
}

feature_cols = list(order.keys())

X_raw = df[feature_cols].astype(str)
encoder = OrdinalEncoder(categories=[order[c] for c in feature_cols])
X = pd.DataFrame(encoder.fit_transform(X_raw), columns=feature_cols)

le = LabelEncoder()
y = le.fit_transform(df["class"])

print("Target classes:", dict(zip(le.classes_, le.transform(le.classes_))))
X.head()

## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print("Train:", X_train.shape, " Test:", X_test.shape)

## 6. Build the Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest trained with", rf.n_estimators, "trees.")

## 7. Evaluate the Model

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
print(f"Train Accuracy: {rf.score(X_train, y_train):.4f}\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=le.classes_, cmap="Blues", ax=ax)
ax.set_title("Random Forest Confusion Matrix")
plt.show()

## 8. Feature Importance
Shows which attributes matter most in deciding car acceptability.

In [ ]:
importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance.round(4))

plt.figure(figsize=(7, 4))
sns.barplot(x=importance.values, y=importance.index, palette="mako")
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.show()

## 9. Compare with a Single Decision Tree
This shows why the ensemble helps.

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_acc = accuracy_score(y_test, dt.predict(X_test))

cv_rf = cross_val_score(rf, X, y, cv=5)
cv_dt = cross_val_score(dt, X, y, cv=5)

compare = pd.DataFrame({
    "Test Accuracy": [dt_acc, acc],
    "5-Fold CV Mean": [cv_dt.mean(), cv_rf.mean()],
    "5-Fold CV Std": [cv_dt.std(), cv_rf.std()]
}, index=["Decision Tree", "Random Forest"]).round(4)
compare

## 10. Effect of Number of Trees

In [ ]:
n_list = [1, 5, 10, 25, 50, 100, 200]
scores = []
for n in n_list:
    m = RandomForestClassifier(n_estimators=n, random_state=42).fit(X_train, y_train)
    scores.append(accuracy_score(y_test, m.predict(X_test)))

plt.figure(figsize=(7, 4))
plt.plot(n_list, scores, marker="o")
plt.title("Accuracy vs Number of Trees")
plt.xlabel("n_estimators")
plt.ylabel("Test Accuracy")
plt.grid(alpha=0.3)
plt.show()

## 11. Predict a New Car

In [ ]:
new_car = pd.DataFrame([["low", "low", "4", "4", "big", "high"]], columns=feature_cols)
new_encoded = pd.DataFrame(encoder.transform(new_car), columns=feature_cols)
pred = rf.predict(new_encoded)
print("Car details:", new_car.iloc[0].to_dict())
print("Predicted acceptability:", le.inverse_transform(pred)[0])

## Conclusion
- All six attributes are categorical, so they were ordinal encoded using their natural order and the class labels were label encoded.
- The Random Forest Classifier reaches high accuracy (typically around 95% or more) on the held-out test data.
- Safety and number of persons are the most influential features, which is sensible: unsafe cars and 2-seaters are mostly unacceptable.
- The Random Forest is more stable than a single Decision Tree, since combining many trees reduces variance and overfitting.
- The dataset is imbalanced (most cars are `unacc`), so minority classes such as `good` and `vgood` have lower recall than the majority class.